# Imports

In [1]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder) without this

/Users/mac/Documents/dev/ID2221/dic/Week 2


In [2]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *
import numpy as np
from Queries import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/16 12:10:03 WARN Utils: Your hostname, MacBook-Pro-som-tillhor-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.247 instead (on interface en0)
26/09/16 12:10:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/mac/Documents/dev/ID2221/dic/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/mac/.ivy2.5.2/cache
The jars for the packages stored in: /Users/mac/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cb77f629-e61f-46b8-b04a-7a00d40d02ea;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-ap

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


# Ensure uncached tables

In [3]:
spark.catalog.uncacheTable("default.air_quality")
spark.catalog.uncacheTable("default.taxi_trips")
spark.catalog.uncacheTable("default.taxi_zone_lookup")
spark.catalog.uncacheTable("default.weather")

# MVT for partition pruning
Let's base our MVT around the default values and on a log scale

In [4]:
partition_bytes = np.array([int(0.5*134217728), 1*134217728, 2*134217728, 4*134217728]) # 64MB, 128MB, 256MB, 512MB
partitions = np.array([100, 200, 400, 800])

# get all combinations of partition_bytes and partitions
from itertools import product
combinations = list(product(partition_bytes, partitions))
print(f'combinations: {combinations}')

for partition_bytes, partitions in combinations:
    print(f'Running with partition_bytes={partition_bytes}, partitions={partitions}')
    spark.conf.set("spark.sql.files.maxPartitionBytes", partition_bytes)
    # spark.conf.set("spark.sql.files.openCostInBytes", partition_bytes)
    # spark.conf.set("spark.sql.files.maxPartitionBytes", partition_bytes)
    # spark.conf.set("spark.sql.files.openCostInBytes", partition_bytes)

    spark.conf.set("spark.sql.shuffle.partitions", partitions)
    for i in range(5):
        print(f'Run {i+1} for partition_bytes={partition_bytes}, partitions={partitions}')
        with log_step(f'Run {i+1} for partition_bytes={partition_bytes}, partitions={partitions}'):
            # Run your queries here
            spark.sql(query_2_1()).show()
            spark.sql(query_2_2()).show()
            spark.sql(query_2_3()).show()


combinations: [(np.int64(67108864), np.int64(100)), (np.int64(67108864), np.int64(200)), (np.int64(67108864), np.int64(400)), (np.int64(67108864), np.int64(800)), (np.int64(134217728), np.int64(100)), (np.int64(134217728), np.int64(200)), (np.int64(134217728), np.int64(400)), (np.int64(134217728), np.int64(800)), (np.int64(268435456), np.int64(100)), (np.int64(268435456), np.int64(200)), (np.int64(268435456), np.int64(400)), (np.int64(268435456), np.int64(800)), (np.int64(536870912), np.int64(100)), (np.int64(536870912), np.int64(200)), (np.int64(536870912), np.int64(400)), (np.int64(536870912), np.int64(800))]
Running with partition_bytes=67108864, partitions=100
Run 1 for partition_bytes=67108864, partitions=100


26/09/16 12:10:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|        Baisley Park|    1|      999|
|  Van Cortlandt Park|    1|       18|
|                SoHo|    1|    20197|
|            Kips Bay|    1|    32984|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142708|
|   Hillcrest/Pomonok|    1|      187|
|   Rossville/Woodrow|    1|        1|
|       Rockaway Park|    1|       80|
|     Oakland Gardens|    1|       56|
|           Stapleton|    1|        4|
|Penn Station/Madi...|    2|        1|
|          Ozone Park|    1|      105|
|            Flushing|    1|      250|
|             Maspeth|    1|      290|
|East New York/Pen...|    1|      245|
|          Ocean Hill|    1|      354|
|          Bath Beach|    1|       58|
|  Claremont/Bathgate|    1|      173|
|          Highbridge|    1|      173|
+--------------------+-----+---------+
only showing top 20 rows


+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|  zero_or_null|2539795|3.6824345463125323|
|greater_than_0| 424773| 3.470743355446981|
+--------------+-------+------------------+



+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
Run 2 for partition_bytes=67108864, partitions=100
+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|        Baisley Park|    1|      999|
|  Van Cortlandt Park|    1|       18|
|                SoHo|    1|    20197|
|            Kips Bay|    1|    32984|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142708|
|   Hillcrest/Pomonok|    1|      187|
|   Ro